# VERA Master Learning Lab: Complete Self-Contained Clinical RAG Pipeline

---

### Objective
This interactive notebook is a standalone, step-by-step tutorial covering the complete architecture of the VERA Clinical Decision Support RAG pipeline:
1. Medical PDF Ingestion and Section-Aware Chunking
2. Dense Embeddings Generation (SentenceTransformers)
3. Vector Indexing (ChromaDB) and Lexical Indexing (BM25)
4. Query Expansion and Hybrid Retrieval (Dense + BM25 with RRF)
5. Safety Confidence Gating and Out-of-Scope Protection
6. Evidence-Grounded Synthesis with Exact In-line Citations

---


## Step 1: Core Dependencies and Imports

The following libraries form the foundation of our clinical RAG pipeline:

| Library | Role in Pipeline |
|:---|:---|
| `pypdf` / `pdfplumber` | Extracts text, pages, and metadata from clinical guideline PDFs. |
| `sentence-transformers` | Generates 384-dimensional dense semantic embeddings using BAAI/bge-small-en-v1.5. |
| `chromadb` | High-performance vector database for semantic nearest-neighbor search. |
| `rank-bm25` | Lexical keyword retrieval for exact pharmaceutical and genomic term matching. |
| `google.generativeai` | LLM synthesis grounded strictly in retrieved medical context. |


In [ ]:
import os
import re
import json
import time
from pathlib import Path

# PDF parsing
import pypdf
import pdfplumber

# Vector database and embeddings
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

# LLM integration
import google.generativeai as genai

print("All dependencies successfully imported.")


## Step 2: Document Ingestion and PDF Parsing

In this step, medical guidelines and clinical trial publications are parsed page by page.
Each page extracts:
- `filename`: Original document title
- `page_number`: 1-indexed page identifier (used for exact citation tracking)
- `text`: Cleaned clinical text content


In [ ]:
PDF_DIR = "../data/raw_pdfs"
if not os.path.exists(PDF_DIR):
    PDF_DIR = "./data/raw_pdfs"

def load_pdf_pages(pdf_path):
    """Loads all pages from a medical PDF and extracts text and page metadata."""
    reader = pypdf.PdfReader(pdf_path)
    filename = os.path.basename(pdf_path)
    pages_data = []
    
    for page_idx, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text_clean = " ".join(text.split())
        if len(text_clean) > 30:
            pages_data.append({
                "doc_name": filename,
                "page_number": page_idx,
                "text": text_clean
            })
    return pages_data

# Load all PDF guidelines from the directory
all_pages = []
pdf_files = list(Path(PDF_DIR).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF guideline documents:")
for pdf_file in pdf_files:
    pages = load_pdf_pages(str(pdf_file))
    all_pages.extend(pages)
    print(f" - {pdf_file.name}: {len(pages)} pages parsed")

print(f"\nTotal parsed pages across all guidelines: {len(all_pages)}")


In [ ]:
# Preview a sample parsed page
if all_pages:
    print("Sample Parsed Page Metadata and Content:")
    print(json.dumps(all_pages[0], indent=2)[:400] + "...")


## Step 3: Section-Aware Text Chunking

### Why Chunking is Necessary
LLM context windows and embedding models perform best on focused, coherent passages.
Chunking divides large pages into overlapping windows (e.g., 500 characters with 100-character overlap) so that:
- Clinical concepts spanning across page boundaries are not lost.
- Precise evidence snippets can be retrieved and cited.


In [ ]:
def split_text_into_chunks(pages, chunk_size=500, chunk_overlap=100):
    """Splits pages into overlapping text chunks while preserving document and page metadata."""
    chunks = []
    chunk_counter = 1
    
    for page_info in pages:
        text = page_info["text"]
        doc_name = page_info["doc_name"]
        page_num = page_info["page_number"]
        
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_content = text[start:end].strip()
            
            if len(chunk_content) > 40:
                chunks.append({
                    "chunk_id": f"chk_{chunk_counter:04d}",
                    "doc_name": doc_name,
                    "page_number": page_num,
                    "content": chunk_content
                })
                chunk_counter += 1
                
            start += (chunk_size - chunk_overlap)
            
    return chunks

all_chunks = split_text_into_chunks(all_pages, chunk_size=500, chunk_overlap=100)
print(f"Created {len(all_chunks)} text chunks ready for vector indexing.")


## Step 4: Semantic Dense Embeddings

Embeddings convert clinical text passages into 384-dimensional dense numerical vectors that represent **semantic meaning and clinical context**, beyond simple keyword matching.
We use the lightweight, high-precision `BAAI/bge-small-en-v1.5` model running locally on CPU.


In [ ]:
print("Loading BAAI/bge-small-en-v1.5 embedding model...")
try:
    embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu", local_files_only=True)
except Exception:
    embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")

print("Embedding model loaded successfully.")

def get_embedding(text):
    """Generates normalized vector embedding for a query or passage."""
    return embed_model.encode(text, normalize_embeddings=True).tolist()

# Test embedding generation on a sample text
sample_vec = get_embedding("Spinal Muscular Atrophy treatment protocol")
print(f"Sample Embedding Dimension: {len(sample_vec)} (First 5 dimensions: {sample_vec[:5]})")


## Step 5: Vector Indexing (ChromaDB) and Lexical Indexing (BM25)

To achieve **Hybrid Retrieval**, we construct two complementary indexes:
1. **ChromaDB**: For semantic dense vector search (capturing synonyms, clinical intent, and context).
2. **BM25**: For exact lexical term matching (capturing precise drug names, gene symbols like *SMN1*, *SMN2*, and dosages).


In [ ]:
# 1. Initialize In-Memory ChromaDB
chroma_client = chromadb.Client()
collection_name = "lab_clinical_guidelines"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# Populate ChromaDB in batches
batch_size = 50
print(f"Indexing {len(all_chunks)} chunks into ChromaDB...")

for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    ids = [c["chunk_id"] for c in batch]
    docs = [c["content"] for c in batch]
    metas = [{"doc_name": c["doc_name"], "page_number": c["page_number"]} for c in batch]
    embeddings = embed_model.encode(docs, normalize_embeddings=True).tolist()
    
    collection.add(
        ids=ids,
        documents=docs,
        embeddings=embeddings,
        metadatas=metas
    )

print(f"Successfully indexed {collection.count()} chunks in ChromaDB.")

# 2. Build BM25 Lexical Index
tokenized_corpus = [c["content"].lower().split() for c in all_chunks]
bm25_index = BM25Okapi(tokenized_corpus)
print(f"BM25 index built with {len(tokenized_corpus)} tokenized documents.")


## Step 6: Clinical Query Expansion

Query expansion enriches the physician inquiry with key clinical synonyms, brand names, and related genomic terms before retrieval.
For example, expanding `"SMA"` with `"Spinal Muscular Atrophy"`, `"SMN1"`, `"SMN2"`, `"Nusinersen"`, and `"Spinraza"`.


In [ ]:
CLINICAL_SYNONYM_MAP = {
    "sma": ["Spinal Muscular Atrophy", "SMN1", "SMN2"],
    "nusinersen": ["Spinraza", "antisense oligonucleotide", "ASO", "intrathecal"],
    "zolgensma": ["onasemnogene abeparvovec", "AAV9 gene therapy"],
    "risdiplam": ["Evrysdi", "small molecule splicing modifier"],
    "chromosomal": ["rearrangements", "translocation", "inversion", "long-read", "nanopore", "PacBio"]
}

def expand_clinical_query(query):
    """Enriches query with relevant clinical synonyms and pharmaceutical terms."""
    expanded_terms = [query]
    q_lower = query.lower()
    
    for key, synonyms in CLINICAL_SYNONYM_MAP.items():
        if key in q_lower:
            expanded_terms.extend(synonyms)
            
    return " ".join(list(dict.fromkeys(expanded_terms)))

# Example expansion
sample_q = "What is the dosing for Nusinersen in SMA?"
print("Original Query:", sample_q)
print("Expanded Query:", expand_clinical_query(sample_q))


## Step 7: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

We combine results from **Dense Vector Search** and **BM25 Lexical Search** using Reciprocal Rank Fusion (RRF):

$$\text{RRF Score}(d) = \sum_{m \in M} \frac{w_m}{k + \text{rank}_m(d)}$$

This ensures documents that perform well in both semantic context and exact keyword matching receive the highest priority.


In [ ]:
def hybrid_retrieve(query, top_k=4, dense_weight=0.7, bm25_weight=0.3):
    """Performs hybrid retrieval combining ChromaDB vector search and BM25 with RRF scoring."""
    expanded_q = expand_clinical_query(query)
    
    # 1. Dense Semantic Search
    q_emb = get_embedding(expanded_q)
    dense_res = collection.query(
        query_embeddings=[q_emb],
        n_results=min(top_k * 2, len(all_chunks)),
        include=["documents", "metadatas", "distances"]
    )
    
    # 2. BM25 Lexical Search
    q_tokens = expanded_q.lower().split()
    bm25_scores = bm25_index.get_scores(q_tokens)
    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k * 2]
    
    # 3. Reciprocal Rank Fusion
    rrf_scores = {}
    chunk_map = {c["chunk_id"]: c for c in all_chunks}
    
    # Process Dense results
    if dense_res and dense_res["ids"]:
        for rank, (cid, dist) in enumerate(zip(dense_res["ids"][0], dense_res["distances"][0])):
            sim = max(0.0, 1.0 - dist)
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (dense_weight / (60 + rank))
            chunk_map[cid]["similarity"] = sim
            
    # Process BM25 results
    for rank, idx in enumerate(top_bm25_indices):
        cid = all_chunks[idx]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (bm25_weight / (60 + rank))
        if "similarity" not in chunk_map[cid]:
            chunk_map[cid]["similarity"] = 0.70
            
    # Sort by combined RRF score
    sorted_cids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    
    results = []
    for cid in sorted_cids:
        ch = chunk_map[cid]
        results.append({
            "chunk_id": ch["chunk_id"],
            "content": ch["content"],
            "metadata": {
                "doc_name": ch["doc_name"],
                "page_number": ch["page_number"]
            },
            "similarity": ch.get("similarity", 0.75),
            "rrf_score": rrf_scores[cid]
        })
        
    return results

# Test hybrid retrieval
test_results = hybrid_retrieve("Nusinersen dosing schedule and loading protocol in SMA", top_k=3)
print(f"Retrieved {len(test_results)} top relevant chunks:")
for i, r in enumerate(test_results, 1):
    print(f"\n[{i}] Source: {r['metadata']['doc_name']} (Page {r['metadata']['page_number']}) | Similarity: {r['similarity']:.3f}")
    print(f"Content: {r['content'][:200]}...")


## Step 8: Safety Confidence Gating and Out-of-Scope Guardrails

In clinical decision support, if the highest retrieval similarity score is below the confidence threshold (`0.60`), the system must safely **refuse to answer** rather than generating unverified assumptions or hallucinations.


In [ ]:
def evaluate_safety_gate(retrieved_chunks, threshold=0.60):
    """Verifies that retrieval confidence exceeds safety thresholds before generation."""
    if not retrieved_chunks:
        return False, 0.0, "No relevant clinical evidence was found in the guidelines."
    
    max_similarity = max(c.get("similarity", 0.0) for c in retrieved_chunks)
    passed = max_similarity >= threshold
    
    if passed:
        reason = f"Passed: Max similarity score {max_similarity:.2f} >= threshold {threshold:.2f}."
    else:
        reason = f"Refusal: Max similarity score {max_similarity:.2f} is below safety threshold {threshold:.2f}."
        
    return passed, max_similarity, reason

# Test with in-scope and out-of-scope queries
passed_in, score_in, reason_in = evaluate_safety_gate(test_results)
print("In-Scope Query Evaluation:")
print(f"Status: {passed_in} | Confidence: {score_in:.2f} | Reason: {reason_in}")

empty_results = []
passed_out, score_out, reason_out = evaluate_safety_gate(empty_results)
print("\nOut-of-Scope Query Evaluation:")
print(f"Status: {passed_out} | Confidence: {score_out:.2f} | Reason: {reason_out}")


## Step 9: Evidence-Grounded LLM Generation and In-line Citations

We construct a strict grounding prompt that commands the LLM to:
1. Answer exclusively from the provided retrieved passages.
2. Structure the answer into an **Executive Summary** followed by **Actionable Recommendations**.
3. Attach exact in-line citations `[Document Name | Page X]` to every clinical assertion.


In [ ]:
def build_clinical_prompt(query, chunks):
    """Builds a structured prompt enforcing strict grounding in retrieved evidence."""
    context_blocks = []
    for idx, c in enumerate(chunks, 1):
        meta = c["metadata"]
        context_blocks.append(
            f"[Source {idx}] Document: {meta['doc_name']} | Page: {meta['page_number']}\n"
            f"Content:\n{c['content']}\n"
        )
    context_text = "\n---\n".join(context_blocks)
    
    prompt = f"""You are VERA (Verified Evidence Retrieval Assistant), a specialized clinical decision-support AI.

### RETRIEVED CLINICAL GUIDELINES:
{context_text}

---

### CLINICAL INQUIRY:
{query}

---

### INSTRUCTIONS:
- Answer the inquiry based SOLELY on the retrieved context above.
- Structure your response into:
  1. **Executive Summary**: A concise direct synthesis answering the question.
  2. **Clinical Recommendations**: 3 to 5 clear bullet points with exact citations [Document | Page X].
- If the retrieved context does not contain sufficient information, state clearly that evidence is insufficient.
"""
    return prompt

def generate_grounded_answer(query, chunks, gemini_api_key=None):
    """Generates an evidence-grounded response using Google Gemini API or deterministic synthesis."""
    passed, score, reason = evaluate_safety_gate(chunks)
    if not passed:
        return f"INSUFFICIENT EVIDENCE: {reason}"
        
    prompt = build_clinical_prompt(query, chunks)
    api_key = gemini_api_key or os.getenv("GEMINI_API_KEY")
    
    if api_key and len(api_key.strip()) >= 15:
        try:
            genai.configure(api_key=api_key)
            model = genai.GenerativeModel("models/gemini-3.1-flash-lite")
            response = model.generate_content(prompt)
            if response and response.text:
                return response.text
        except Exception as e:
            print(f"Gemini API call note: {e}")
            
    # Deterministic offline fallback synthesis
    top = chunks[0]
    return (
        f"**Executive Summary:**\n"
        f"Based on the retrieved clinical evidence for '{query}', recommendations indicate: "
        f"{top['content'][:250]}... [{top['metadata']['doc_name']} | Page {top['metadata']['page_number']}].\n\n"
        f"**Clinical Recommendations:**\n"
        f"• \"{top['content'][:200]}...\" [{top['metadata']['doc_name']} | Page {top['metadata']['page_number']}].\n"
        f"• Consult the referenced institutional guideline for comprehensive dosage and administration protocols."
    )


## Step 10: End-to-End Interactive Clinical RAG Pipeline

Run a live query through the complete VERA RAG pipeline to observe all steps in action:
1. Query Classification and Expansion
2. Hybrid Dense + Lexical Retrieval
3. Safety Confidence Gating
4. Grounded Synthesis and Citation Formatting


In [ ]:
# Set your clinical inquiry here
CLINICAL_QUERY = "What are the initiation criteria and dosing considerations for Nusinersen in SMA patients?"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

print("=" * 70)
print(f"CLINICAL INQUIRY: {CLINICAL_QUERY}")
print("=" * 70)

# Step 1: Hybrid Retrieval
retrieved = hybrid_retrieve(CLINICAL_QUERY, top_k=4)
print(f"\nStep 1: Retrieved {len(retrieved)} evidence chunks.")

# Step 2: Safety Gate Evaluation
passed, score, reason = evaluate_safety_gate(retrieved)
print(f"Step 2: Safety Gate -> Status: {'PASSED' if passed else 'FAILED'} (Confidence: {score:.2f})")

# Step 3: Grounded Synthesis
print("\nStep 3: Generating Grounded Synthesis...\n")
final_answer = generate_grounded_answer(CLINICAL_QUERY, retrieved, gemini_api_key=GEMINI_API_KEY)

print("-" * 70)
print(final_answer)
print("-" * 70)

print("\nStep 4: Verified Source Citations:")
for idx, r in enumerate(retrieved, 1):
    print(f" [{idx}] {r['metadata']['doc_name']} (Page {r['metadata']['page_number']})")
print("=" * 70)
